## Clasificación de tópicos en reviews (Ollama + Llama 3)

Experimento: leer `reviews_clean.csv`, y para cada review identificar uno o más **tópicos** entre opciones predefinidas (u otros si no encajan), junto con el **fragmento** de la review que sustenta cada tópico.

**Requisitos:** [Ollama](https://ollama.com) en ejecución local y el modelo `llama3` instalado (`ollama pull llama3`).

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

try:
    import ollama
except ImportError:
    raise ImportError("Instala el cliente: pip install ollama pandas")

# Ruta al CSV (notebook en Tesis/notebooks/)
DATA_PATH = Path(r"../sources/data/reviews_clean.csv").resolve()
OUT_PATH = Path(r"../sources/data/reviews_topicos_llama3.csv").resolve()

MODELO = "llama3"

TOPICOS_PREDEFINIDOS = [
    "Limpieza",
    "Atención al cliente",
    "Instalaciones y servicios",
    "Relación calidad-precio",
    "Comodidad",
    "Desayuno / Gastronomía",
]

TOPICOS_TEXTO = "\n".join(f"- {t}" for t in TOPICOS_PREDEFINIDOS)
print("CSV:", DATA_PATH.exists(), DATA_PATH)
print("Modelo:", MODELO)

CSV: True C:\Estudio\Maestria\Tesis\sources\data\reviews_clean.csv
Modelo: llama3


### Prompt y llamada al modelo

Se pide salida **JSON** con `(topico, fragmento, sentimiento_topico)`. El **sentimiento_topico** es la polaridad del fragmento respecto a ese aspecto (positivo / negativo / neutro). Si el sentimiento global del dataset es `neutro`, igualmente hay que valorar el tono hacia cada tópico en su fragmento. Si un tópico predefinido no encaja, el modelo propone un nombre corto alternativo en `topico`.

In [2]:
SYSTEM_PROMPT = """Eres un analista de opiniones de hoteles. Tu tarea es leer una review en español y listar los TÓPICOS que menciona o implica.

Tópicos predefinidos (usa el nombre EXACTO si la review encaja):
{topicos}

Reglas:
- Una misma review puede tener VARIOS tópicos.
- Para cada tópico, copia un FRAGMENTO literal o casi literal de la review (cita breve) que justifique ese tópico.
- Para cada par tópico–fragmento, indica sentimiento_topico: la valoración que expresa ese fragmento respecto a ESE aspecto (no el sentimiento global de toda la review).
- Valores permitidos para sentimiento_topico: "positivo", "negativo" o "neutro". Usa "neutro" solo si el fragmento es factualmente equilibrado o no muestra clara satisfacción ni insatisfacción sobre ese aspecto.
- Si te damos un sentimiento global de referencia (p. ej. neutro), aun así debes asignar positivo/negativo/neutro por tópico según el fragmento; las reviews neutras suelen mezclar aspectos buenos y malos.
- Si el contenido no encaja bien en ninguno de los predefinidos, inventa un nombre breve y claro para el tópico (por ejemplo: "Ruido / descanso", "Ubicación", "Parking").
- No inventes fragmentos: deben aparecer en la review o ser un recorte mínimo fiel.
- Responde SOLO con un objeto JSON válido, sin markdown ni texto fuera del JSON.

Formato obligatorio:
{{"items": [{{"topico": "...", "fragmento": "...", "sentimiento_topico": "positivo"}}, ...]}}
""".format(topicos=TOPICOS_TEXTO)


def _normalizar_sentimiento_topico(val) -> str | None:
    if val is None or (isinstance(val, str) and not val.strip()):
        return None
    s = str(val).strip().lower()
    if s in ("positivo", "positive", "pos"):
        return "positivo"
    if s in ("negativo", "negative", "neg"):
        return "negativo"
    if s in ("neutro", "neutral"):
        return "neutro"
    return None


def parsear_respuesta_json(texto: str) -> list[dict]:
    """Extrae items del JSON devuelto por el modelo (tolerante a ruido)."""
    texto = texto.strip()
    # A veces el modelo envuelve en ```json
    m = re.search(r"\{[\s\S]*\}", texto)
    if not m:
        return []
    blob = m.group(0)
    data = json.loads(blob)
    items = data.get("items", [])
    if not isinstance(items, list):
        return []
    out = []
    for it in items:
        if not isinstance(it, dict):
            continue
        top = str(it.get("topico", "")).strip()
        frag = str(it.get("fragmento", "")).strip()
        st = _normalizar_sentimiento_topico(it.get("sentimiento_topico"))
        if top and frag:
            out.append({"topico": top, "fragmento": frag, "sentimiento_topico": st})
    return out


def extraer_uso_tokens(resp: dict) -> dict:
    """Campos que devuelve Ollama en cada respuesta (no siempre presentes si hay caché, etc.)."""
    p = resp.get("prompt_eval_count")
    e = resp.get("eval_count")
    total = None
    if isinstance(p, int) and isinstance(e, int):
        total = p + e
    return {
        "prompt_eval_count": p,
        "eval_count": e,
        "total_tokens": total,
    }


def clasificar_review(
    texto_review: str,
    *,
    sentimiento_global: str | None = None,
    return_usage: bool = False,
):
    bloques = [f"Review:\n{texto_review}"]
    if sentimiento_global is not None and str(sentimiento_global).strip():
        bloques.append(
            "Sentimiento global etiquetado en el dataset (referencia, no sustituye el análisis por fragmento): "
            + str(sentimiento_global).strip()
        )
    user = "\n\n".join(bloques)
    r = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        format="json",
        options={"temperature": 0.2},
    )
    content = r["message"]["content"]
    items = parsear_respuesta_json(content)
    if return_usage:
        return items, extraer_uso_tokens(r)
    return items

### Cargar datos y probar con una fila

In [3]:
df = pd.read_csv(DATA_PATH)
assert "review" in df.columns, "Se espera columna 'review'"
print(df.shape)
df

(8076, 2)


,review,sentimiento
0,El fin de semana mi pareja y yo hicimos una re...,negativo
1,"El hotel en general está bien, las habtiacione...",neutro
2,"El hotel es moderno, amplio y limpio, pero no ...",neutro
3,[PERSONA] averiada o no operativa. Se comenta ...,negativo
4,Este hotel ha bajado notoriamente su categoria...,negativo
...,...,...
8071,Pero el wifi era muy lento. El parqueadero fue...,neutro
8072,"Buena experiencia en generaallo recomiendo, au...",neutro
8073,"Me gustó el servicio, la verdad poca variedad ...",neutro
8074,Aunque igual el aire acondicionado hacia ruido...,neutro


In [4]:
# Iniciar Ollama desde Python (requiere que Ollama esté instalado)
import subprocess
import sys

def iniciar_ollama():
    """
    Intenta iniciar Ollama en modo servidor si no está corriendo.
    """
    import socket

    def is_ollama_running(host='localhost', port=11434):
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            return False

    if not is_ollama_running():
        print("Ollama no está corriendo. Intentando iniciar Ollama...")
        try:
            if sys.platform.startswith("win"):
                # En Windows, buscar ollama.exe en PATH
                subprocess.Popen("ollama serve", shell=True, creationflags=subprocess.DETACHED_PROCESS)
            else:
                # En Unix/macOS
                subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            import time
            # Esperar unos segundos para que arranque
            for _ in range(10):
                if is_ollama_running():
                    print("Ollama iniciado exitosamente.")
                    break
                time.sleep(1)
            else:
                print("No se pudo iniciar Ollama automáticamente. Por favor, inícialo manualmente.")
        except Exception as ex:
            print(f"Error al intentar iniciar Ollama: {ex}")

iniciar_ollama()

Ollama no está corriendo. Intentando iniciar Ollama...
Ollama iniciado exitosamente.


In [5]:
# Prueba rápida (requiere Ollama corriendo)
ejemplo = df["review"].iloc[3]
print(ejemplo)
sent = df["sentimiento"].iloc[3] if "sentimiento" in df.columns else None
items, uso = clasificar_review(str(ejemplo), sentimiento_global=sent, return_usage=True)
print(json.dumps(items, ensure_ascii=False, indent=2))
print("Tokens (respuesta Ollama):", uso)
# prompt_eval_count = tokens de entrada evaluados; eval_count = tokens generados

[PERSONA] averiada o no operativa. Se comenta en recepción y no me hacen caso. reitero y tampoco ni caso. Pasé frío. No es admisible en ningún hotel pero menos en un hotel de esta supuesta categoría
[
  {
    "topico": "Atención al cliente",
    "fragmento": "Se comenta en recepción y no me hacen caso. reitero y tampoco ni caso.",
    "sentimiento_topico": "negativo"
  },
  {
    "topico": "Comodidad",
    "fragmento": "Pasé frío.",
    "sentimiento_topico": "negativo"
  }
]
Tokens (respuesta Ollama): {'prompt_eval_count': 537, 'eval_count': 80, 'total_tokens': 617}


### Procesar lote (ajusta `N_MAX` o `OFFSET`)

- `N_MAX=None` procesa todo el dataset (puede tardar mucho).
- Los resultados se guardan en columnas serializadas JSON: `topicos_json` (lista de objetos) y `topicos_resumen` (solo nombres de tópico, separados por `|`).
- Por interacción, Ollama informa `prompt_eval_count` (tokens de entrada), `eval_count` (salida) y `total_tokens` (suma si ambos vienen informados).

In [6]:
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

N_MAX = 50   # Pon None para todas las filas
OFFSET = 50

sub = df.iloc[OFFSET : OFFSET + N_MAX] if N_MAX is not None else df.iloc[OFFSET:]
rows_out = []

for idx, row in tqdm(sub.iterrows(), total=len(sub)):
    text = str(row["review"])
    sent_row = row["sentimiento"] if "sentimiento" in row else None
    try:
        items, uso = clasificar_review(
            text, sentimiento_global=sent_row, return_usage=True
        )
    except Exception as e:
        items = [
            {
                "topico": "_error",
                "fragmento": str(e),
                "sentimiento_topico": None,
            }
        ]
        uso = {"prompt_eval_count": None, "eval_count": None, "total_tokens": None}
    nombres = [x["topico"] for x in items]
    rows_out.append({
        "idx_original": idx,
        "sentimiento": row["sentimiento"] if "sentimiento" in row else None,
        "topicos_json": json.dumps(items, ensure_ascii=False),
        "topicos_resumen": " | ".join(nombres),
        "n_topicos": len(items),
        "prompt_eval_count": uso["prompt_eval_count"],
        "eval_count": uso["eval_count"],
        "total_tokens": uso["total_tokens"],
    })

resultado = pd.DataFrame(rows_out)
resultado.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("Guardado:", OUT_PATH)
resultado.head(10)

  0%|          | 0/50 [00:00<?, ?it/s]

Guardado: C:\Estudio\Maestria\Tesis\sources\data\reviews_topicos_llama3.csv


,idx_original,sentimiento,topicos_json,topicos_resumen,n_topicos,prompt_eval_count,eval_count,total_tokens
0,50,neutro,"[{""topico"": ""Desayuno / Gastronomía"", ""fragmen...",Desayuno / Gastronomía | Atención al cliente |...,4,585,180,765
1,51,neutro,"[{""topico"": ""Limpieza"", ""fragmento"": ""Tiene ol...",Limpieza,1,533,39,572
2,52,neutro,"[{""topico"": ""Instalaciones y servicios"", ""frag...",Instalaciones y servicios | Atención al client...,4,631,189,820
3,53,negativo,"[{""topico"": ""Comodidad"", ""fragmento"": ""las cam...",Comodidad | Instalaciones y servicios | Desayu...,6,675,320,995
4,54,negativo,"[{""topico"": ""Limpieza"", ""fragmento"": ""la minin...",Limpieza | Instalaciones y servicios | Atenció...,5,644,236,880
5,55,neutro,"[{""topico"": ""Comodidad"", ""fragmento"": ""la habi...",Comodidad | Instalaciones y servicios,2,683,110,793
6,56,negativo,"[{""topico"": ""Atención al cliente"", ""fragmento""...",Atención al cliente | Limpieza | Instalaciones...,4,583,154,737
7,57,negativo,"[{""topico"": ""Instalaciones y servicios"", ""frag...",Instalaciones y servicios | Comodidad | Comodi...,5,698,281,979
8,58,negativo,"[{""topico"": ""Instalaciones y servicios"", ""frag...",Instalaciones y servicios | Instalaciones y se...,4,703,208,911
9,59,neutro,"[{""topico"": ""Limpieza"", ""fragmento"": ""seguido ...",Limpieza | Atención al cliente | Instalaciones...,4,626,188,814


### Inspección: expandir `topicos_json` (una fila por tópico–fragmento–`sentimiento_topico`)

In [7]:
def expandir_topicos(df_res: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for _, r in df_res.iterrows():
        items = json.loads(r["topicos_json"])
        for it in items:
            filas.append({
                "idx_original": r["idx_original"],
                "sentimiento_global_dataset": r.get("sentimiento"),
                "topico": it["topico"],
                "fragmento": it["fragmento"],
                "sentimiento_topico": it.get("sentimiento_topico"),
            })
    return pd.DataFrame(filas)


expandir_topicos(resultado).head(20) if len(resultado) else None

,idx_original,sentimiento_global_dataset,topico,fragmento,sentimiento_topico
0,50,neutro,Desayuno / Gastronomía,Desayuno para un 4 estrellas debería haber más...,negativo
1,50,neutro,Atención al cliente,Ni siquiera nos han preguntado en el Check out...,negativo
2,50,neutro,Atención al cliente,En el check in le cobran directamente pero por...,negativo
3,50,neutro,Instalaciones y servicios,"Reformas necesita el hotel, zonas muy degradadas",negativo
4,51,neutro,Limpieza,Tiene olor a rancio,negativo
5,52,neutro,Instalaciones y servicios,la moqueta está regular la decoración obsoleta...,negativo
6,52,neutro,Atención al cliente,"el personal es estupendo, muy amable y agradable",positivo
7,52,neutro,Desayuno / Gastronomía,el desayuno es bueno,positivo
8,52,neutro,Relación calidad-precio,tienen precios adecuados y moderados. Relación...,positivo
9,53,negativo,Comodidad,las camas son viejas y mejor no te muevas porq...,negativo


In [8]:
# Calculamos el total de tokens usados sumando la columna "total_tokens"
if "total_tokens" in resultado.columns:
    total_tokens_usados = resultado["total_tokens"].dropna().sum()
    print("Total de tokens usados:", int(total_tokens_usados))
else:
    print("No hay columna 'total_tokens' para calcular uso de tokens.")

Total de tokens usados: 38129


In [9]:
# Sumamos cada métrica por separado
if all(col in resultado.columns for col in ["prompt_eval_count", "eval_count", "total_tokens"]):
    total_prompt_eval_count = resultado["prompt_eval_count"].dropna().sum()
    total_eval_count = resultado["eval_count"].dropna().sum()
    total_tokens = resultado["total_tokens"].dropna().sum()
    print("Total prompt_eval_count:", int(total_prompt_eval_count))
    print("Total eval_count:", int(total_eval_count))
    print("Total tokens:", int(total_tokens))
else:
    print("Alguna de las columnas no está presente: 'prompt_eval_count', 'eval_count', 'total_tokens'")

Total prompt_eval_count: 30180
Total eval_count: 7949
Total tokens: 38129
